In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# import FCI code
from causallearn.search.ConstraintBased.FCI import fci
from causallearn.utils.cit import fisherz
from causallearn.utils.GraphUtils import GraphUtils

#set project root as .../recidivism-causal
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set causal pitfalls root
nij_root = project_root / "data" / "processed"
#test it works
nij_root

output_dir = project_root/"results"/"NIJ"/"graphs_fci"
output_dir.mkdir(parents=True,exist_ok=True)

In [12]:
#path to dataset
csv_path = nij_root/"NIJ_compact_WIP.csv"

#load into df and sanity check form
df = pd.read_csv(csv_path)
df.head(), df.shape
X = df.to_numpy(dtype=float)
X_small=X[:,:10]

g_small, _ = fci(X_small, fisherz, alpha=0.05, verbose=True, depth=2, max_path_length = 2)  #causal graph object
out_name = f"NIJ__FCI_SMALL.png"
out_path = output_dir / out_name

# Save PNG using GraphUtils
py_dot = GraphUtils.to_pydot(g_small, labels=list(df.columns[:10]))
py_dot.write_png(str(out_path))

  0%|          | 0/10 [00:00<?, ?it/s]

0 dep 1 | () with p-value 0.000416

0 ind 2 | () with p-value 0.301425

0 ind 3 | () with p-value 0.079335

0 dep 4 | () with p-value 0.000044

0 ind 5 | () with p-value 0.078278

0 dep 6 | () with p-value 0.000000

0 ind 7 | () with p-value 0.468503

0 dep 8 | () with p-value 0.011827

0 ind 9 | () with p-value 0.054018

1 dep 0 | () with p-value 0.000416

1 dep 2 | () with p-value 0.000000

1 dep 3 | () with p-value 0.002756

1 dep 4 | () with p-value 0.000000

1 dep 5 | () with p-value 0.000000

1 dep 6 | () with p-value 0.000000

1 ind 7 | () with p-value 0.657283

1 dep 8 | () with p-value 0.000000

1 dep 9 | () with p-value 0.008785

2 ind 0 | () with p-value 0.301425

2 dep 1 | () with p-value 0.000000

2 dep 3 | () with p-value 0.000000

2 dep 4 | () with p-value 0.000000

2 dep 5 | () with p-value 0.014180

2 dep 6 | () with p-value 0.000027

2 dep 7 | () with p-value 0.000081

2 dep 8 | () with p-value 0.000000

2 dep 9 | () with p-value 0.000000

3 ind 0 | () with p-value 0.

In [19]:
import time
#lean first pass subset...
demo_cols = [
    "Gender_F",
    "Race_BLACK",
    "Age_at_Release_18-22",
    "Age_at_Release_23-27",
    "Age_at_Release_28-32",
    "Age_at_Release_33-37",
    "Age_at_Release_38-42",
    "Age_at_Release_43-47",
    "Age_at_Release_48 or older",
]

criminal_core_cols = [
    "Gang_Affiliated",
    "Prior_Arrest_Episodes_Felony_high",
    "Prior_Arrest_Episodes_Property_high",
    "Prior_Arrest_Episodes_Drug_high",
    "Prior_Conviction_Episodes_Felony_high",
]

dynamic_core_cols = [
    "Percent_Days_Employed",
    "Jobs_Per_Year",
    "Avg_Days_per_DrugTest",
    "DrugTests_Cocaine_Positive",   #one drug indicator chosen... 
    "Delinquency_Reports_high",
]

risk_score_cols = [
    "Supervision_Risk_Score_First",
]

outcome_cols = [
    "Recidivism_Within_3years",
]

lean_cols = (
    demo_cols
    + criminal_core_cols
    + dynamic_core_cols
    + risk_score_cols
    + outcome_cols
)

# Make sure all exist in df
lean_cols = [c for c in lean_cols if c in df.columns]

print("Using", len(lean_cols), "columns:")
print(lean_cols)

df_lean = df[lean_cols].copy()
X_lean = df_lean.to_numpy(dtype=float)

start = time.time() 
g_lean, _ = fci(
    X_lean,
    fisherz,
    alpha=0.05,
    depth=1,           # keep shallow for tractability
    max_path_length=1, # strict first pass
    verbose=True,
)
end = time.time()

#draw graph

out_name = "NIJ_FCI_lean_first_pass.png"
out_path = output_dir / out_name

py_dot_lean = GraphUtils.to_pydot(g_lean, labels=list(df_lean.columns))
py_dot_lean.write_png(str(out_path))


Using 21 columns:
['Gender_F', 'Race_BLACK', 'Age_at_Release_18-22', 'Age_at_Release_23-27', 'Age_at_Release_28-32', 'Age_at_Release_33-37', 'Age_at_Release_38-42', 'Age_at_Release_43-47', 'Age_at_Release_48 or older', 'Gang_Affiliated', 'Prior_Arrest_Episodes_Felony_high', 'Prior_Arrest_Episodes_Property_high', 'Prior_Arrest_Episodes_Drug_high', 'Prior_Conviction_Episodes_Felony_high', 'Percent_Days_Employed', 'Jobs_Per_Year', 'Avg_Days_per_DrugTest', 'DrugTests_Cocaine_Positive', 'Delinquency_Reports_high', 'Supervision_Risk_Score_First', 'Recidivism_Within_3years']


  0%|          | 0/21 [00:00<?, ?it/s]

0 dep 1 | () with p-value 0.000000

0 dep 2 | () with p-value 0.000000

0 dep 3 | () with p-value 0.000024

0 ind 4 | () with p-value 0.301968

0 dep 5 | () with p-value 0.005413

0 dep 6 | () with p-value 0.000001

0 dep 7 | () with p-value 0.000000

0 ind 8 | () with p-value 0.554554

0 dep 9 | () with p-value 0.000326

0 dep 10 | () with p-value 0.001200

0 dep 11 | () with p-value 0.000000

0 dep 12 | () with p-value 0.000000

0 dep 13 | () with p-value 0.005550

0 ind 14 | () with p-value 0.097205

0 ind 15 | () with p-value 0.999658

0 dep 16 | () with p-value 0.000000

0 ind 17 | () with p-value 0.377664

0 dep 18 | () with p-value 0.000000

0 ind 19 | () with p-value 0.709965

0 dep 20 | () with p-value 0.000000

1 dep 0 | () with p-value 0.000000

1 dep 2 | () with p-value 0.000000

1 dep 3 | () with p-value 0.000000

1 ind 4 | () with p-value 0.778420

1 dep 5 | () with p-value 0.003624

1 dep 6 | () with p-value 0.000000

1 dep 7 | () with p-value 0.000000

1 dep 8 | () with

In [21]:
print(f"FCI run depth =1, max_path_length=1 took {(end - start)/60:.2f} minutes")

FCI run depth =1, max_path_length=1 took 1.75 minutes
